# Phase 4 — Cross-Country Eval on Colab (T4)

Runs Faster R-CNN's zero-shot cross-country evaluation on GPU. YOLOv8's cross-country
results are already complete (done locally on CPU).

### This is EVAL only, not training
The Faster R-CNN checkpoint (`experiments/runs/faster_rcnn_source/best.pt`) was already
trained in the Phase 3a notebook. This run just loads it and scores it on each TARGET
country, zero-shot.

### Resumable
`cross_country_eval.py` skips any (model, dataset) pair that's already logged in
`experiment_log.csv` with a matching config hash — `source` and `czech` were already
evaluated locally (on CPU, before switching to Colab for speed) and are bundled in, so
this run picks up straight at `norway`.

### Three changes since the bundle was first built — read before running

1. **A real bug is fixed.** `--device cuda` used to be silently ignored by the actual
   detection inference (it only affected a tiny latency micro-benchmark) — every prior
   evaluation ran on CPU regardless of what device was requested. That's fixed now, so
   this run will actually use the GPU for the bulk of the work, not just claim to.
2. **China_Drone is dropped from this evaluation.** YOLOv8 already has a complete
   China_Drone result (2,401 images, mAP@0.5=0.2107) computed before this change — that
   row stays in `experiment_log.csv`, but Faster R-CNN will never get a matching one, so
   any side-by-side model comparison table has 5 target countries for YOLOv8 and 4 for
   Faster R-CNN. State that asymmetry if you use the comparison in the report.
3. **Norway is evaluated on a 2,000-image SEEDED RANDOM subsample**, not the full 8,161
   — chosen randomly (not "first 2,000", which would bias toward one drive
   route/session since RDD2022 filenames are capture-sequence ordered), and reproducible
   (fixed seed). This is *not* the same 8,161 images YOLOv8's existing Norway number was
   computed on — again, worth a footnote wherever these two numbers are compared directly.

The bundle already on Drive still has China_Drone's image files in it — that's harmless,
this run just never touches them; no need to rebuild or re-upload anything.

## Step 1 — Confirm a GPU is attached

Fails deliberately if not — CPU here defeats the entire point of this notebook.

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU, then rerun."
print("device:", torch.cuda.get_device_name(0))

## Step 2 — Mount Google Drive

See the Phase 3a notebook if this fails (`MessageError: credential propagation was unsuccessful` is a browser third-party-cookie issue, not an account problem). Step 2b below is a no-Drive fallback.

In [ ]:
from google.colab import drive
if not __import__('pathlib').Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

## Step 2b — FALLBACK ONLY: upload the bundle directly if Drive mount fails

In [ ]:
# FALLBACK ONLY -- skip if Step 2 succeeded.
from google.colab import files
uploaded = files.upload()          # select rdd_bundle.zip
BUNDLE_ZIP = '/content/' + next(iter(uploaded))
print("uploaded to:", BUNDLE_ZIP)

## Step 3 — Unpack the bundle

This bundle is DIFFERENT from the Phase 3a training bundle: it carries TARGET country
data (Norway is ~8.2 GB by itself), the already-trained Faster R-CNN checkpoint, and the
current `experiment_log.csv` (so the resume-skip in Step 6 actually has something to
match against). Expect the extraction to take a few minutes given the size.

In [ ]:
import zipfile, os, time
from pathlib import Path

try:
    BUNDLE_ZIP
except NameError:
    BUNDLE_ZIP = '/content/drive/MyDrive/rdd_bundle.zip'
print("using bundle:", BUNDLE_ZIP)

REPO = Path('/content/road-damage-detection')
SENTINEL = REPO / 'src' / 'eval' / 'cross_country_eval.py'

if not SENTINEL.exists():
    print("extracting (this bundle includes Norway's 8+ GB -- can take a few minutes)...")
    REPO.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(BUNDLE_ZIP) as zf:
        zf.extractall(REPO)
    print(f"extracted in {time.time() - t0:.0f}s")
else:
    print("code already extracted, skipping unpack")

assert SENTINEL.exists(), f"{SENTINEL} still missing -- is rdd_bundle.zip the Phase 4 eval bundle?"
os.chdir(REPO)
print("cwd:", os.getcwd())

## Step 4 — Install dependencies

In [ ]:
!pip install -q ultralytics
import ultralytics
print("ultralytics", ultralytics.__version__)

## Step 5 — Point the dataset configs at Colab paths

Same reason as Phase 3a: the YAMLs ship with Windows absolute paths and must be
regenerated for wherever they're actually running.

In [ ]:
!python src/data/write_dataset_configs.py --data-root /content/road-damage-detection/data/processed

## Step 6 — Verify the checkpoint and target data arrived, and check what's already logged

The checkpoint file existing is what makes evaluation possible at all; per-country image
counts should match `data/split_report.json` (czech 2829, norway 8161, us 4805,
china_motorbike 1977). China_Drone is deliberately NOT checked here — this evaluation
run doesn't touch it (see the note above), even though its files are still in the bundle.
The last part shows what Step 7 will skip.

In [ ]:
from pathlib import Path

weights = Path('experiments/runs/faster_rcnn_source/best.pt')
assert weights.exists(), f"{weights} missing -- rebuild the bundle with --include-weights faster_rcnn_source"
print(f"checkpoint OK: {weights} ({weights.stat().st_size / 1e6:.1f} MB)")

print()
for country in ['czech', 'norway', 'us', 'china_motorbike']:
    imgs = len(list(Path(f'data/processed/target/{country}/images').glob('*.jpg')))
    print(f"  {country:16} {imgs:5d} images")
print("\n  (norway will be evaluated on a 2,000-image random subsample -- see the note above)")

In [ ]:
import csv
from pathlib import Path

log_path = Path('experiments/results/experiment_log.csv')
if log_path.exists():
    rows = list(csv.DictReader(open(log_path)))
    fr_rows = [r for r in rows if r['run_id'].startswith('faster_rcnn_source_eval_on_')]
    print(f"Already logged ({len(fr_rows)} rows) -- Step 7 will skip these:")
    for r in fr_rows:
        print(f"  {r['dataset']:16} mAP@0.5={r['map50']}  F1={r['f1']}")
else:
    print("No experiment_log.csv found -- Step 7 will evaluate everything from scratch.")

## Step 7 — Run the cross-country evaluation

`--models faster_rcnn` only -- YOLOv8's cross-country results are already complete
(with China_Drone included, from before it was dropped). This run should resume-skip
`source` and `czech` (already done) and evaluate `norway` (2,000-image subsample), `us`,
`china_motorbike` fresh, genuinely on GPU now that `--device cuda` actually reaches the
inference calls.

Progress and the results CSV are both written incrementally per dataset (not just at the
end), so if a session drops mid-run, rerunning this exact cell picks back up rather than
starting over -- the same lesson chunked training already applied, now applied here too.

In [ ]:
!python src/eval/cross_country_eval.py --models faster_rcnn --device cuda

## Step 8 — Review the results

In [ ]:
import pandas as pd
df = pd.read_csv('experiments/results/cross_country_results.csv')
pd.set_option('display.width', 200)
cols = ['model','dataset','kind','num_images','map50','map50_pct_change_vs_indomain','f1','f1_pct_change_vs_indomain']
print(df[cols].to_string(index=False))

## Step 9 — Save everything back to Drive

Copies `experiments/runs` (failure-example images) and `experiments/results`
(`cross_country_results.csv`, updated `experiment_log.csv`) to Drive. Falls back to a
browser download if Drive was never mounted (Step 2b path).

In [ ]:
import shutil, os
from pathlib import Path

def save_results():
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        out = drive_root / 'rdd_results_phase4'
        out.mkdir(parents=True, exist_ok=True)
        for src in [Path('experiments/runs'), Path('experiments/results')]:
            if src.exists():
                dst = out / src.name
                if dst.exists():
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
                print("saved to Drive:", dst)
        return out
    print("Drive not mounted -- packing results for download instead.")
    archive = shutil.make_archive('/content/rdd_results_phase4', 'zip', 'experiments')
    print("archive:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
    from google.colab import files
    files.download(archive)
    return Path(archive)

out = save_results()
print("\nPhase 4 (Faster R-CNN) complete. Bring cross_country_results.csv and "
      "experiment_log.csv back to the local repo's incoming/ folder for merging.")